# Russian River Step 3 -- 2D mesh

Form the 2D mesh, elevate via a DEM, condition.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
# setting up logging first or else it gets preempted by another package
import watershed_workflow.io
watershed_workflow.io.setupLogging(1)

In [ ]:
import os,sys
import logging
import numpy as np
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import pickle
import shapely
import pandas as pd
import geopandas as gpd
pd.options.display.max_columns = None
import copy

import watershed_workflow 
import watershed_workflow.utils
import watershed_workflow.sources
import watershed_workflow.mesh
import watershed_workflow.plot
import watershed_workflow.sources.standard_names

# set the default figure size for notebooks
plt.rcParams["figure.figsize"] = (8, 6)

## Input: Parameters and other source data

In [ ]:
# Force Watershed Workflow to pull data from this directory rather than a shared data directory.
# This picks up the Coweeta-specific datasets set up here to avoid large file downloads for 
# demonstration purposes.
#
def splitPathFull(path):
    """
    Splits an absolute path into a list of components such that
    os.path.join(*splitPathFull(path)) == path
    """
    parts = []
    while True:
        head, tail = os.path.split(path)
        if head == path:  # root on Unix or drive letter with backslash on Windows (e.g., C:\)
            parts.insert(0, head)
            break
        elif tail == path:  # just a single file or directory
            parts.insert(0, tail)
            break
        else:
            parts.insert(0, tail)
            path = head
    return parts

cwd = splitPathFull(os.getcwd())
assert cwd[-1] == 'workflow'
cwd = cwd[:-1]

# Note, this directory is where downloaded data will be put as well
data_dir = os.path.join(*(cwd + ['input_data',]))
def toInput(filename):
    return os.path.join(data_dir, filename)

output_dir = os.path.join(*(cwd + ['output_data',]))
output_filenames = dict()
def fromOutput(filename):
    return os.path.join(output_dir, filename)    

def toOutput(role, filename):
    output_filenames[role] = filename
    return fromOutput(filename)

# check output and input dirs exist
if not os.path.isdir(data_dir):
    os.makedirs(data_dir, exist_ok=True)
if not os.path.isdir(output_dir):
    os.makedirs(output_dir, exist_ok=True)
       

In [ ]:
# Set the data directory to the local space to get the locally downloaded files
# REMOVE THIS CELL for general use outside fo Coweeta
watershed_workflow.utils.setDataDirectory(data_dir)


In [ ]:
## Parameters cell -- this provides all parameters that can be changed via pipelining to generate a new watershed. 
name = 'RussianRiver'
hucs = ['18010110'] # a list of HUCs to run


# -- parameters to clean and reduce the river network prior to meshing
prune_by_area = 20               # km^2
simplify = 200                   # length scale to target average edge 

# -- mesh triangle refinement control
refine_d0 = 200
refine_d1 = 600

refine_L0 = 200
refine_L1 = 500

refine_A0 = refine_L0**2 / 2
refine_A1 = refine_L1**2 / 2


# Refine triangles if they get too acute
min_angle = 20 # degrees

# width of reach by stream order (order:width)
river_widths = dict({1:10, 2:10, 3:20, 4:30, 5:30}) 


# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs = watershed_workflow.crs.default_crs

## Reload data

In [ ]:
with open(fromOutput('03a_watersheds.pickle'), 'rb') as fid:
    watersheds = pickle.load(fid)

reaches = gpd.read_parquet(fromOutput('03b_rivers.parquet'))
rivers = watershed_workflow.hydro.createRivers(reaches, method='native')


In [ ]:
# load m2 mesh from file -- we will use this for testing pitfilling algorithms
with open(fromOutput('03b_m2_preconditioning_smoothed.pickle'), 'rb') as fid:
    m2_orig = pickle.load(fid)

In [ ]:
assert not np.any(np.isnan(m2_orig.coords))

In [ ]:

print('2D labeled sets')
print('---------------')
for ls in m2_orig.labeled_sets:
    print(f'{ls.setid} : {ls.entity} : {len(ls.ent_ids)} : "{ls.name}"')

In [ ]:
# fill pits away from the river
outlet_edge_ls = next(ls for ls in m2_orig.labeled_sets if 'outlet' == ls.name and 'FACE' == ls.entity)
assert len(outlet_edge_ls.ent_ids) == 1
outlet_edge = outlet_edge_ls.ent_ids[0]
preserved_pits = [c for (c,conn) in enumerate(m2_orig.conn) if len(conn) > 3]

In [ ]:
fig, axs = plt.subplots(2,2, figsize=(12,10))

pits = watershed_workflow.mesh.findPits(m2_orig, preserved_pits=preserved_pits, forced_outlet_edges=[outlet_edge,])
watershed_workflow.mesh.plotWorstPit(m2_orig, pits, context_rings=5)
plt.show()

In [ ]:
# fill pits away from the river
outlet_edge_ls = next(ls for ls in m2_orig.labeled_sets if 'outlet' == ls.name and 'FACE' == ls.entity)
assert len(outlet_edge_ls.ent_ids) == 1
outlet_edge = outlet_edge_ls.ent_ids[0]
preserved_pits = [c for (c,conn) in enumerate(m2_orig.conn) if len(conn) > 3]

m2 = copy.deepcopy(m2_orig)
m2r, res = watershed_workflow.mesh.conditionMesh(m2,
                                                 preserved_pits=preserved_pits,
                                                 forced_outlet_edges=[outlet_edge,],
                                                 epsilon = 0.01
                                                )

plt.show()

In [ ]:
res[-1]['pits_final']


In [ ]:
# double-check...
pits = watershed_workflow.mesh.findPits(m2r,
                                             forced_outlet_edges=[outlet_edge,])
for pit in pits:
    print(pit)

In [ ]:
fig, axs = plt.subplots(1,3, sharex=True, sharey=True, figsize=(10,5))

img1 = m2_orig.plot(facecolors='elevation', cmap='gist_earth', ax=axs[0])
axs[0].set_title('Original')
img2 = m2r.plot(facecolors='elevation', cmap='gist_earth', ax=axs[1])
axs[1].set_title('Refined and Pit Filled')

# interpolate orig onto m2 for differencing
import scipy.interpolate
z_orig = scipy.interpolate.griddata(m2_orig.coords[:,0:2], m2_orig.coords[:,2], m2r.centroids[:,0:2], method='linear')
diff = m2r.centroids[:,2] - z_orig

img3 = m2r.plot(facecolors=diff, cmap='RdBu_r', ax=axs[2], vmin=diff.min(), vmax=diff.max())
axs[2].set_title('difference')

watersheds.plot(ax=axs[0], color='k')
for river in rivers:
    river.plot(ax=axs[0], color='r')
m2_orig.plot(ax=axs[0], edgecolor='grey', linewidth=0.5, facecolor='none', colorbar=False)
    
watersheds.plot(ax=axs[1], color='k')
for river in rivers:
    river.plot(ax=axs[1], color='r')
m2r.plot(ax=axs[1], edgecolor='grey', linewidth=0.5, facecolor='none', colorbar=False)

watersheds.plot(ax=axs[2], color='k')
for river in rivers:
    river.plot(ax=axs[2], color='r')
m2r.plot(ax=axs[2], edgecolor='grey', linewidth=0.5, facecolor='none', colorbar=False)

#output = widgets.Output()
output = None
scaler = watershed_workflow.plot.DynamicColormapScaler(output=output)
scaler.connect(axs[0])
scaler.addGroup([img1, img2], False)
scaler.addGroup([img3,], True)

#display(output)
#for ax, img in zip(axs, [img1, img2, img3]):
#    ax.set_aspect('equal', adjustable='box')
#    divider = make_axes_locatable(ax)
#    cax = divider.append_axes("right", size="5%", pad=0.05)
#    plt.colorbar(img, cax=cax)
    
fig.canvas.draw()
plt.tight_layout()
plt.show()

In [ ]:
m2r.cell_data['elevation'] = m2r.centroids[:,-1]
m = m2r.to_dataframe().explore('elevation', tiles="Esri.WorldImagery")
m = watershed_workflow.makeMap(m)
m

In [ ]:
print('2D labeled sets')
print('---------------')
for ls in m2r.labeled_sets:
    print(f'{ls.setid} : {ls.entity} : {len(ls.ent_ids)} : "{ls.name}"')

In [ ]:
# pre-partition the mesh 
print(m2r.num_cells)
print(m2r.num_cells * 10 / 4000)

# NERSC nodes
print(m2r.num_cells * 10 / 4000 / 128)
#
## let's use 8 nodes * 128 cores per
m2p = m2r.partition(128 * 8, True)

In [ ]:
# save the mesh and regions
with open(toOutput('m2', '03c_m2.pickle'), 'wb') as fid:
    pickle.dump(m2p, fid)

In [ ]:
np.save(toOutput('m2_centroids', 'm2_centroids.npy'), m2r.centroids)
np.save(toOutput('m2_vertices', 'm2_vertices.npy'), m2r.coords)

In [ ]:
# save output filenames
with open(toOutput('03c_output_filenames', '03c_output_filenames.txt'), 'wb') as fid:
    pickle.dump(output_filenames, fid)